In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

In [2]:
df = pd.read_parquet("D:\Estudos\DeepLearning\inicioTratamentoCancer\dadosFinal.parquet")

In [3]:
df["atraso60Binario"] = df["atraso60"].map({
    "Pontual": 0,
    "Atraso": 1
})

y = df["atraso60Binario"]

X = df.drop(columns=[
    "atraso60",
    "atraso60Binario"
])

In [4]:
categoricas = X.select_dtypes(include=['object', 'category']).columns
numericas   = X.select_dtypes(exclude=['object', 'category']).columns

In [5]:
pipeCategoricas = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value = 'Desconhecido')), # Transforma NA em uma coluna própria
    ('onehot', OneHotEncoder(handle_unknown = 'ignore')) # Transforma as colunas em números para a RF conseguir interpretar. handle unknown ignora colunas desconhecidas
])

pipeNumericas = Pipeline([
    ('imputer', IterativeImputer(
        estimator = HistGradientBoostingRegressor(random_state=100), # Histogram-based Gradient Boosting Regression Tree. Rápido! Agrupa números em histogramas e faz gradient boosting com árvores de decisão
        add_indicator = True, # Apesar de imputação, a informação de que aquela variável era originalmente NA continua
        random_state = 100
    ))
])

preprocessador = ColumnTransformer([ # Aplica cada imputer nas variáveis corretas
    ('cat', pipeCategoricas, categoricas),
    ('num', pipeNumericas, numericas)
])

arvoreEstocastica = Pipeline([
    ('preprocessador', preprocessador),
    ('rf', RandomForestClassifier(
        n_jobs = -1,
        random_state = 100
    ))
])

In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3, random_state = 100, stratify = y)

In [ ]:
arvoreEstocastica.fit(X_train, y_train) #Treinando o modelo
pred = arvoreEstocastica.predict_proba(X_test)[:,1]
print("Acurácia: ", roc_auc_score(y_test, pred))

In [ ]:
amostra = X_test.sample(10000, random_state=100).index

resultadoPermutacao = permutation_importance(
    arvoreEstocastica,
    X_test.loc[amostra],
    y_test.loc[amostra],
    n_repeats = 3,
    random_state = 100,
    n_jobs = 1
)

In [ ]:
nomeVariavel = ( # Nomes das colunas depois do OneHotEncoder
    arvoreEstocastica.named_steps['preprocessador']
    .get_feature_names_out()
)

featureImportance = pd.DataFrame({ # Cria o DataFrame com os nomes e valores de importância associados
    "variavel": X_test.columns,
    "importancia": resultadoPermutacao.importances_mean
})

featureImportance["nome"] = ( # Limpa os nomes das variáveis
    featureImportance["variavel"]
    .str.replace("cat__", "", regex=False)
    .str.replace("num__", "", regex=False)
)

fiTopo = featureImportance.sort_values(by="importancia", ascending=False).head(20)

fiTopo = fiTopo.sort_values(by="importancia", ascending=True)

plt.figure(figsize=(10, 8))
plt.barh(fiTopo["nome"], fiTopo["importancia"], color='skyblue')
plt.xlabel("Aumento no Erro (Permutation Importance)")
plt.title("20 Variáveis mais relevantes para o atraso no tratamento")
plt.tight_layout()
plt.show()